In [1]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
# Import necessary libraries
from transformers import MarianMTModel, MarianTokenizer
import gradio as gr

# Define the model name
model_name = 'Helsinki-NLP/opus-mt-en-fr'

# Load the tokenizer and model
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def translate(text):
    # Tokenize the input text
    tokenized_text = tokenizer.prepare_seq2seq_batch([text], return_tensors='pt')
    
    # Perform the translation
    translation = model.generate(**tokenized_text)
    
    # Decode the translated text
    translated_text = tokenizer.decode(translation[0], skip_special_tokens=True)
    
    return translated_text

# Create a Gradio interface
iface = gr.Interface(
    fn=translate, 
    inputs=gr.Textbox(label="Enter text to translate"), 
    outputs=gr.Textbox(label="Translated text"),
    title="English to French Translator",
    description="Enter English text and get the French translation."
)

# Launch the Gradio interface
iface.launch()

/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [2]:
trust_remote_code=True

In [3]:
from datasets import load_dataset
iwslt = load_dataset("IWSLT/iwslt2017", "iwslt2017-en-fr")

print(iwslt)

/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 232825
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 8597
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 890
    })
})


In [5]:
# Preview
sample = iwslt['test'][0, 1]
print(sample)


{'translation': [{'en': 'Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.', 'fr': "Il y a plusieurs années, ici à Ted, Peter Skillman a présenté une épreuve de conception appelée l'épreuve du marshmallow."}, {'en': "And the idea's pretty simple:  Teams of four have to build the tallest free-standing structure  out of 20 sticks of spaghetti,  one yard of tape, one yard of string  and a marshmallow.", 'fr': "Et l'idée est plutôt simple. Des équipes de quatre personnes doivent bâtir la plus haute structure tenant debout avec 20 spaghettis, un mètre de ruban collant, un mètre de ficelle, et un marshmallow."}]}


In [4]:
opensub = load_dataset("Helsinki-NLP/open_subtitles", lang1="en", lang2="fr")

In [24]:
opensub

DatasetDict({
    train: Dataset({
        features: ['id', 'meta', 'translation'],
        num_rows: 41763488
    })
})

In [5]:
opensub_sample = opensub['train'].select(range(1000))  # 50k pairs

# Check the structure of opensub dataset
print("OpenSubtitles sample:")
sample = opensub_sample[0]
print(sample)
print("\nTranslation type:", type(sample["translation"]))
if isinstance(sample["translation"], dict):
    print("Translation keys:", list(sample["translation"].keys()))
    print("Translation values:")
    for key, value in sample["translation"].items():
        print(f"  {key}: {value[:100]}...")  # First 100 chars
elif isinstance(sample["translation"], list):
    print("Translation is a list with", len(sample["translation"]), "items")
    if len(sample["translation"]) > 0:
        print("First item keys:", list(sample["translation"][0].keys()) if isinstance(sample["translation"][0], dict) else "Not a dict")

opensub

OpenSubtitles sample:
{'id': '0', 'meta': {'year': 0, 'imdbId': 1089124, 'subtitleId': {'en': 4995691, 'fr': 4588599}, 'sentenceIds': {'en': [1], 'fr': [1]}}, 'translation': {'en': 'I never dreamed before', 'fr': "I've never dreamed before I'm gonna knock the door"}}

Translation type: <class 'dict'>
Translation keys: ['en', 'fr']
Translation values:
  en: I never dreamed before...
  fr: I've never dreamed before I'm gonna knock the door...


DatasetDict({
    train: Dataset({
        features: ['id', 'meta', 'translation'],
        num_rows: 41763488
    })
})

In [6]:
iwslt_sample = iwslt["train"].select(range(1000))

In [7]:
import pandas as pd
from langdetect import detect, LangDetectException

def to_df(ds, src_lang="en", tgt_lang="fr", dataset_name=""):
    data = []
    swap_count = 0
    skip_count = 0
    
    # First, check if we need to swap keys by analyzing multiple samples
    sample_size = min(20, len(ds))
    en_is_french = 0
    fr_is_french = 0
    
    for i in range(sample_size):
        row = ds[i]
        tr = row["translation"]
        if isinstance(tr, dict) and 'en' in tr and 'fr' in tr:
            try:
                en_detected = detect(tr['en'])
                fr_detected = detect(tr['fr'])
                if en_detected == 'fr':
                    en_is_french += 1
                if fr_detected == 'fr':
                    fr_is_french += 1
            except LangDetectException:
                pass
    
    # Determine if keys are swapped based on sample analysis
    keys_swapped = en_is_french > fr_is_french
    if keys_swapped:
        print(f"Note: Detected that OpenSubtitles dataset has swapped keys. 'en' contains French, 'fr' contains English.")
        print(f"  (Checked {sample_size} samples: {en_is_french} had French in 'en', {fr_is_french} had French in 'fr')")
    
    for row in ds:
        tr = row["translation"]
        # Handle different dataset structures
        # IWSLT: translation is a list of dicts
        if isinstance(tr, list):
            # For IWSLT, process each item in the list
            for item in tr:
                if isinstance(item, dict) and src_lang in item and tgt_lang in item:
                    data.append({"src": item[src_lang], "tgt": item[tgt_lang]})
        # OpenSubtitles: translation is a dict
        elif isinstance(tr, dict):
            if 'en' in tr and 'fr' in tr:
                # Swap if we determined keys are swapped
                if keys_swapped:
                    src_text = tr['fr']  # 'fr' key actually has English
                    tgt_text = tr['en']  # 'en' key actually has French
                else:
                    src_text = tr['en']
                    tgt_text = tr['fr']
                
                # Skip if both appear to be English (data quality issue)
                try:
                    src_detected = detect(src_text)
                    tgt_detected = detect(tgt_text)
                    if src_detected == 'en' and tgt_detected == 'en':
                        skip_count += 1
                        continue  # Skip this row
                except LangDetectException:
                    pass  # Can't detect, include anyway
                
                data.append({"src": src_text, "tgt": tgt_text})
            elif src_lang in tr and tgt_lang in tr:
                # Fallback to original keys
                data.append({"src": tr[src_lang], "tgt": tr[tgt_lang]})
            else:
                # Debug: print what keys we actually have (only first time)
                if len(data) == 0:
                    print(f"Warning: Could not find expected keys. Available keys: {list(tr.keys())}")
    
    if skip_count > 0:
        print(f"Note: Skipped {skip_count} rows where both texts appeared to be English")
    
    return pd.DataFrame(data)

df_iwslt = to_df(iwslt_sample, dataset_name="IWSLT")
df_open = to_df(opensub_sample, dataset_name="OpenSubtitles")
print("IWSLT head:")
print(df_iwslt.head())
print("\nOpenSubtitles head:")
print(df_open.head())
print("\nSample OpenSubtitles row:")
if len(df_open) > 0:
    print(f"src: {df_open.iloc[0]['src']}")
    print(f"tgt: {df_open.iloc[0]['tgt']}")

Note: Skipped 2 rows where both texts appeared to be English
Note: Skipped 52 rows where both texts appeared to be English
IWSLT head:
                                                 src  \
0  Thank you so much, Chris. And it's truly a gre...   
1  I have been blown away by this conference, and...   
2  And I say that sincerely, partly because  Put ...   
3           I flew on Air Force Two for eight years.   
4  Now I have to take off my shoes or boots to ge...   

                                                 tgt  
0  Merci beaucoup, Chris. C'est vraiment un honne...  
1  J'ai été très impressionné par cette conférenc...  
2  Et je dis çà sincèrement, en autres parce que ...  
3       J'ai volé avec Air Force 2 pendant huit ans.  
4  Et maintenant, je dois enlever mes chaussures ...  

OpenSubtitles head:
                                        src  \
0                    I never dreamed before   
1  "C-Major Pentatonic Scale" It's no good.   
2                       I can't lear

In [15]:
print(df_iwslt.head(20))

                                                  src  \
0   Thank you so much, Chris. And it's truly a gre...   
1   I have been blown away by this conference, and...   
2   And I say that sincerely, partly because  Put ...   
3            I flew on Air Force Two for eight years.   
4   Now I have to take off my shoes or boots to ge...   
5   I'll tell you one quick story to illustrate wh...   
6     It's a true story -- every bit of this is true.   
7   Soon after Tipper and I left the --  White Hou...   
8                                  Driving ourselves.   
9   I know it sounds like a little thing to you, b...   
10                 There was no motorcade back there.   
11  You've heard of phantom limb pain?  This was a...   
12  We were on I-40. We got to Exit 238, Lebanon, ...   
13  We got off the exit, we found a Shoney's resta...   
14  Low-cost family restaurant chain, for those of...   
15  We went in and sat down at the booth, and the ...   
16  And she said "Yes, that's f

In [8]:
def make_context_windows(df, window_size=3):
    """Create (context, current_utterance, target) triples."""
    samples = []
    for i in range(window_size, len(df)):
        context = " ".join(df["src"].iloc[i - window_size:i])
        current = df["src"].iloc[i]
        target = df["tgt"].iloc[i]
        samples.append({
            "context": context,
            "current": current,
            "src_with_context": context + " " + current,
            "tgt": target
        })
        print("context: ", context)
    return pd.DataFrame(samples)

df_open.head(2)

# df_context = make_context_windows(df_open, window_size=3)
# df_context.head(1)


,src,tgt
0,I never dreamed before,I've never dreamed before I'm gonna knock the ...
1,"""C-Major Pentatonic Scale"" It's no good.",Gamme pentatonique en do majeur


In [9]:
def add_prev_tgt(df):
    """
    Create a dataframe with columns: prev_tgt, src, tgt
    where prev_tgt is the previous utterance in the target language.
    
    Args:
        df: DataFrame with 'src' and 'tgt' columns
        
    Returns:
        DataFrame with 'prev_tgt', 'src', 'tgt' columns
    """
    # Create a copy to avoid modifying the original
    result_df = df.copy()
    
    # Shift the 'tgt' column by 1 to get the previous target
    # This will make prev_tgt[i] = tgt[i-1]
    result_df['prev_tgt'] = result_df['tgt'].shift(1)
    
    # Reorder columns: prev_tgt, src, tgt
    result_df = result_df[['prev_tgt', 'src', 'tgt']]
    
    return result_df

# Example usage:
df_iwslt_with_prev = add_prev_tgt(df_iwslt)
print("IWSLT with prev_tgt:")
print(df_iwslt_with_prev.head(10))
print("\nFirst row prev_tgt (should be NaN):", df_iwslt_with_prev.iloc[0]['prev_tgt'])
print("Second row prev_tgt:", df_iwslt_with_prev.iloc[1]['prev_tgt'])


IWSLT with prev_tgt:
                                            prev_tgt  \
0                                               None   
1  Merci beaucoup, Chris. C'est vraiment un honne...   
2  J'ai été très impressionné par cette conférenc...   
3  Et je dis çà sincèrement, en autres parce que ...   
4       J'ai volé avec Air Force 2 pendant huit ans.   
5  Et maintenant, je dois enlever mes chaussures ...   
6  --Rires Applaudissements-- Je vais vous racont...   
7   C'est une histoire vraie, dans tous ses détails.   
8  Après que Tipper et moi avons quitté la --Faux...   
9                             conduisant nous-mêmes.   

                                                 src  \
0  Thank you so much, Chris. And it's truly a gre...   
1  I have been blown away by this conference, and...   
2  And I say that sincerely, partly because  Put ...   
3           I flew on Air Force Two for eight years.   
4  Now I have to take off my shoes or boots to ge...   
5  I'll tell you one quick

In [10]:
df_open_with_prev = add_prev_tgt(df_open)

df_open_with_prev.head()

,prev_tgt,src,tgt
0,None,I never dreamed before,I've never dreamed before I'm gonna knock the ...
1,I've never dreamed before I'm gonna knock the ...,"""C-Major Pentatonic Scale"" It's no good.",Gamme pentatonique en do majeur
2,Gamme pentatonique en do majeur,I can't learn this.,"C'est trop compliqué, je ne peux pas retenir ça."
3,"C'est trop compliqué, je ne peux pas retenir ça.",It's too hard for me.,"Inutile, c'est sans espoir..."
4,"Inutile, c'est sans espoir...",Fool!,Crétin !


In [11]:
df_complete = pd.concat([df_iwslt_with_prev, df_open_with_prev])
df_complete.shape

(1946, 3)

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from datasets import load_dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add context separator token
special_tokens = {"additional_special_tokens": ["</ctx>"]}
tokenizer.add_special_tokens(special_tokens)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))


/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(59515, 512, padding_idx=59513)

In [13]:
def preprocess(examples):
    # Handle batched inputs - each field is a list
    # Handle None/NaN values for prev_tgt (use empty string if None)
    prev_tgts = [str(prev) if prev is not None and str(prev) != 'nan' else "" 
                 for prev in examples["prev_tgt"]]
    
    # Create input strings by concatenating prev_tgt, separator, and src
    input_texts = [
        (prev_tgt + " </ctx> " + src).strip() if prev_tgt else src
        for prev_tgt, src in zip(prev_tgts, examples["src"])
    ]
    
    # Tokenize inputs
    inputs = tokenizer(
        input_texts,
        max_length=256,
        truncation=True,
        padding=True
    )

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["tgt"],
            max_length=256,
            truncation=True,
            padding=True
        )["input_ids"]

    # Replace pad tokens with -100 so they are ignored by the loss
    labels = [
        [(lbl if lbl != tokenizer.pad_token_id else -100) for lbl in label_seq]
        for label_seq in labels
    ]

    inputs["labels"] = labels
    return inputs

In [14]:
df_val_iwslt = to_df(iwslt["validation"], dataset_name="IWSLT")
df_val_open = to_df(opensub["train"].select(range(500, 1000)), dataset_name="OpenSubtitles")

Note: Skipped 2 rows where both texts appeared to be English
Note: Skipped 9 rows where both texts appeared to be English


In [15]:
df_val_iwslt_with_prev = add_prev_tgt(df_val_iwslt)
df_val_open_with_prev = add_prev_tgt(df_val_open)

df_val_complete = pd.concat([df_val_iwslt_with_prev, df_val_open_with_prev])
df_val_complete.shape
df_val_complete.head()

,prev_tgt,src,tgt
0,None,Last year I showed these two slides so that d...,"L'année dernière, je vous ai présenté ces deux..."
1,"L'année dernière, je vous ai présenté ces deux...",But this understates the seriousness of this p...,Mais ceci tend à amoindrir le problème parce q...
2,Mais ceci tend à amoindrir le problème parce q...,"The arctic ice cap is, in a sense, the beatin...",On peut voir la calotte glacière arctique comm...
3,On peut voir la calotte glacière arctique comm...,It expands in winter and contracts in summer.,Elle s'étend en hiver et se contracte en été.
4,Elle s'étend en hiver et se contracte en été.,The next slide I show you will be a rapid fas...,La prochaine diapositive que je vais vous mont...


In [16]:
from datasets import Dataset, DatasetDict
train_dataset = Dataset.from_pandas(df_complete)
val_dataset = Dataset.from_pandas(df_val_complete)

final_dataset = DatasetDict({
    "train": train_dataset,
    "val": val_dataset
})

final_dataset

DatasetDict({
    train: Dataset({
        features: ['prev_tgt', 'src', 'tgt', '__index_level_0__'],
        num_rows: 1946
    })
    val: Dataset({
        features: ['prev_tgt', 'src', 'tgt', '__index_level_0__'],
        num_rows: 1379
    })
})

In [17]:
processed_dataset = final_dataset.map(preprocess, batched=True, remove_columns=final_dataset["train"].column_names)

Map:   0%|          | 0/1946 [00:00<?, ? examples/s]/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 1379/1379 [00:00<00:00, 4205.62 examples/s]


In [18]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./model_out",
    per_device_train_batch_size=4,   # adjust for memory
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    num_train_epochs=3,
    fp16=False,             # DON'T use fp16 (unsupported on Mac GPUs)
    bf16=False,             # bf16 also unsupported
    fp16_opt_level="O0",
    eval_accumulation_steps=2,
    predict_with_generate=True,
    report_to="none",
)

In [19]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(59515, 512, padding_idx=59513)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(59515, 512, padding_idx=59513)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [20]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["val"]
)

trainer.train()

/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,1.799800
1000,1.199600


/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[59513]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1461, training_loss=1.312021974828617, metrics={'train_runtime': 197.7215, 'train_samples_per_second': 29.526, 'train_steps_per_second': 7.389, 'total_flos': 332695502585856.0, 'train_loss': 1.312021974828617, 'epoch': 3.0})

In [21]:
def translate(src_en, prev_fr=""):
    input_text = f"{prev_fr} </ctx> {src_en}"

    enc = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True
    ).to("mps")

    out = model.generate(**enc, max_length=128)

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [23]:
print(translate(
    src_en="When did you see me?",
    prev_fr="Je t'ai vu au café tout à l'heure."   # previous French context
))

Quand m'avez-vous vue ?


In [24]:
def generate_waitk(src_en, prev_fr, k=3):
    # tokenize full source
    full = tokenizer(src_en, return_tensors="pt")
    input_ids = full["input_ids"][0]

    # autoregressive decoding
    output_ids = []
    t = 0

    while True:
        # visible source = first k + t tokens
        visible_src = input_ids[: min(len(input_ids), k + t)]

        batch = tokenizer(
            prev_fr + "</ctx>" + tokenizer.decode(visible_src),
            return_tensors="pt",
            truncation=True
        ).to("mps")

        out = model.generate(
            **batch,
            max_new_tokens=1,
            do_sample=False
        )

        token = out[0][-1].item()
        if token == tokenizer.eos_token_id:
            break

        output_ids.append(token)
        t += 1

    return tokenizer.decode(output_ids, skip_special_tokens=True)
